In [6]:
!pip uninstall numpy pandas -y
!pip install numpy==1.26.4 pandas

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.3.3
Uninstalling pandas-2.3.3:
  Successfully uninstalled pandas-2.3.3


You can safely remove it manually.
You can safely remove it manually.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached pandas-2.3.3-cp39-cp39-win_amd64.whl.metadata (19 kB)
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/15.8 MB 12.2 MB/s eta 0:00:02
   ----------- ---------------------------- 4.5/15.8 MB 10.8 MB/s eta 0:00:02
   ---------------- ----------------------- 6.6/15.8 MB 11.2 MB/s eta 0:00:01
   ----------------------- ---------------- 9.2/15.8 MB 11.0 MB/s eta 0:00:01
   ---------------------------- ----------- 11.3/15.8 MB 10.7 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.8 MB 10.4 MB/s eta 0:00:01
   -------------------------------------- - 15.2/15.8 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------  15.7/15.8 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 8.5 MB/s eta 0:00:00
Using cached pandas-2.3.3-cp39-cp39-win_amd64.whl (11.4 MB)


In [1]:
import os
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [12]:
import os
import shutil

# 配置
base_dir = '../result/archive-92'  # 你的根目录名
datasets = ['adult', 'caltech', 'facebook', 'youtube']
target_seed = '0'
backup_folder_name = 'legacy_backup'

for dataset in datasets:
    dataset_path = os.path.join(base_dir, dataset)
    if not os.path.exists(dataset_path):
        continue
    
    # 遍历 num 层 (如 100, 111 等)
    for num_dir in os.listdir(dataset_path):
        num_path = os.path.join(dataset_path, num_dir)
        if not os.path.isdir(num_path):
            continue
            
        # 定位备份文件夹
        backup_path = os.path.join(num_path, backup_folder_name)
        # 定位目标 seed=0 文件夹
        seed_path = os.path.join(num_path, target_seed)
        
        if os.path.exists(backup_path) and os.path.isdir(backup_path):
            os.makedirs(seed_path, exist_ok=True)
            
            files = os.listdir(backup_path)
            for f in files:
                # 关键判断：如果不以 EfficientBFS 开头，就移回去
                if not f.startswith("EfficientBFS"):
                    src_file = os.path.join(backup_path, f)
                    dst_file = os.path.join(seed_path, f)
                    
                    shutil.move(src_file, dst_file)
                    print(f"已还原: {f} -> {seed_path}")

print("\n还原完成。非 EfficientBFS 开头的文件已回到 seed=0 目录。")

已还原: BFSTC-ub0-d-10.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-11.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-12.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-13.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-14.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-15.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-6.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-7.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-8.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub0-d-9.0-0.95-AdultIncomeFeatureSelection.pckl -> ../result/archive-92\adult\111\0
已还原: BFSTC-ub2-d-6.0-0.95-AdultIncomeFeature

In [13]:
# ================= 配置区 =================
# 包含老数据 (如 archive-old) 和新数据 (如 archive-27, 28) 的路径
ROOT_DIRS = [
    "../result/archive-92", # 替换为你老数据的实际文件夹名
    "../result/archive-94"
] 
OUTPUT_CSV = "master_experiment_results_PowerBI.csv"
# ==========================================

def process_multiple_archives(root_paths):
    data_records = []
    
    for root_path in root_paths:
        root_path = Path(root_path)
        if not root_path.exists():
            print(f"⚠️ 跳过不存在的目录: {root_path}")
            continue
            
        pckl_files = list(root_path.rglob("*.pckl"))
        print(f"🔍 正在解析目录 {root_path.name}: 找到 {len(pckl_files)} 个文件...")
        
        for pckl_file in tqdm(pckl_files, desc=f"Parsing {root_path.name}"):
            try:
                # 1. 提取环境参数
                seed_str = pckl_file.parent.name
                num_str = pckl_file.parent.parent.name
                task_name = pckl_file.parent.parent.parent.name
                
                # 2. 动态解析文件名
                filename = pckl_file.name.replace('.pckl', '')
                parts = filename.split('-')
                
                # 初始化默认值
                is_ls = False
                is_dive = False
                strategy_clean = "Legacy" # 老数据默认策略名
                
                # --- 区分新旧版本文件名 ---
                flag = False
                if len(parts) == 6:
                    # 老格式: Algorithm-Heuristic-Sorting-Budget-Alpha-Model
                    # 例子: Efficient-ub0-d-6.0-0.95-YoutubeCoverage
                    alg = parts[0]
                    heuristic = parts[1]
                    sorting = parts[2]
                    budget = float(parts[3])
                    alpha = float(parts[4])
                    model_name = parts[5]
                    
                    if alg == 'BFSTC':
                        flag = True
                elif len(parts) >= 7 and parts[0] == 'EfficientBFS':
                    # 新格式: EfficientBFS-Strategy[_LS][_Dive]-Heuristic-Sorting-Budget-Alpha-Model
                    alg = parts[0]
                    strategy_raw = parts[1]
                    heuristic = parts[2]
                    sorting = parts[3]
                    budget = float(parts[4])
                    alpha = float(parts[5])
                    model_name = parts[6]
                    
                    is_ls = '_LS' in strategy_raw
                    is_dive = '_Dive' in strategy_raw
                    strategy_clean = strategy_raw.replace('_LS', '').replace('_Dive', '')
                else:
                    # 无法识别的文件，跳过
                    continue
                
                # 3. 读取结果
                with open(pckl_file, 'rb') as f:
                    res = pickle.load(f)
                
                if flag:
                    continue
                
                # 4. 组装记录
                record = {
                    'Archive': root_path.name,
                    'Task': task_name,
                    'Ground_Size': int(num_str),
                    'Seed': int(seed_str),
                    'Algorithm': alg,          # 会区分出 EfficientBFS, Efficient, BFSTC
                    'Strategy': strategy_clean, # 新数据为具体策略，老数据为 Legacy
                    'Local_Search': is_ls,
                    'Dive_Bound': is_dive,
                    'Heuristic': heuristic,
                    'Sorting': sorting,
                    'Budget': budget,
                    'Alpha': alpha,
                    'Model': model_name,
                    
                    'Objective_f(S)': res.get('f(S)', None),
                    'Cost_c(S)': res.get('c(S)', None),
                    'Time_s': res.get('time', None),
                    'Node_Count': res.get('node_count', None),
                    'Open_List_Count': res.get('open_list_count', None),
                    'TLE': res.get('TLE', False), # 老数据的超时标记，新数据没有则默认 False
                    'Solution_Set_Size': len(res.get('S', [])) if 'S' in res else 0 
                }
                data_records.append(record)
                
            except Exception as e:
                print(f"❌ 解析出错 {pckl_file.name}: {e}")

    # 5. 合并导出
    df = pd.DataFrame(data_records)
    
    if not df.empty:
        df.sort_values(by=['Task', 'Budget', 'Algorithm', 'Strategy', 'Local_Search', 'Dive_Bound', 'Seed'], inplace=True)
        df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        print(f"\n✅ 数据合并完毕！共提取 {len(df)} 条有效记录。")
        print(f"💾 已保存至: {OUTPUT_CSV}")
    else:
        print("\n⚠️ 未提取到任何有效数据。")
        
    return df

# 执行合并
df_master = process_multiple_archives(ROOT_DIRS)

🔍 正在解析目录 archive-92: 找到 479 个文件...


Parsing archive-92: 100%|██████████| 479/479 [00:00<00:00, 1404.69it/s]


🔍 正在解析目录 archive-94: 找到 884 个文件...


Parsing archive-94: 100%|██████████| 884/884 [00:00<00:00, 2013.64it/s]


✅ 数据合并完毕！共提取 1331 条有效记录。
💾 已保存至: master_experiment_results_PowerBI.csv


In [15]:
import shutil
import os

# ================= 自动化推送 OneDrive 配置 =================
source_file = OUTPUT_CSV 

# 使用 ~ 符号，Python 会自动帮你解析出当前电脑的 C:\Users\你的用户名
# 这样写永远不会因为用户名不对而报错
onedrive_dir = os.path.expanduser(r"~\OneDrive - Nanyang Technological University")
onedrive_destination = os.path.join(onedrive_dir, "master_experiment_results_PowerBI.csv")
# ==========================================================

try:
    if os.path.exists(source_file):
        # 1. 确保 subopt 文件夹存在
        os.makedirs(onedrive_dir, exist_ok=True)
        
        # 2. 复制并覆盖
        shutil.copy2(source_file, onedrive_destination)
        
        print(f"\n🚀 起飞！文件已自动推送到: {onedrive_destination}")
    else:
        print("\n⚠️ 找不到源文件，推送失败。")
except Exception as e:
    print(f"\n❌ 推送出错: {e}")


🚀 起飞！文件已自动推送到: C:\Users\LEJIAN001\OneDrive - Nanyang Technological University\master_experiment_results_PowerBI.csv
